## Análisis Gramatical de Reviews de Películas

Este notebook contiene el análisis gramatical y el procesamiento de reviews de películas de un dataset.  

In [ ]:
import nltk
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
from typing import DefaultDict, List, Tuple, Set, Hashable, Iterable, Dict
from collections import Counter, defaultdict
from tqdm import tqdm
from nltk import (
  CFG,  # Context-Free Grammar 
  PCFG  # Probabilistic Context-Free Grammar
)
from nltk.parse import (
  ChartParser,    # Chart Parsing Algorithm
  ViterbiParser   # Viterbi Parsing Algorithm
)
from nltk.tokenize import (
  word_tokenize,  # Tokenize a sentence into words
  sent_tokenize,  # Tokenize text into sentences
)
from spacy import displacy

try:
  nltk.data.find('tokenizers/punkt')
except LookupError:
  nltk.download('punkt')

nlp = spacy.load("es_core_news_sm")


### Cargar Dataset y Extraer Reviews

In [ ]:
def load_reviews(file_path: str, k: int) -> List[str]:
    """
    Carga el dataset y extrae las primeras k reseñas de la columna 'review_text'.
    
    Args:
        file_path (str): Ruta al archivo CSV.
        k (int): Número de reseñas a tomar.
    
    Returns:
        List[str]: Lista con las reseñas.
    """
    df = pd.read_csv(file_path)
    reviews = df['review_text'].head(k).tolist()
    return reviews

# Especificar la ruta del dataset y el número de reseñas K
DATASET_PATH = "../data/film_reviews_result_clean.csv"
K = 10  # Puedes ajustar este valor

reviews = load_reviews(DATASET_PATH, K)
print(f"Se cargaron {len(reviews)} reseñas.")


### Tokenizar Reviews en Oraciones

In [ ]:
import re

def split_reviews_into_sentences(reviews: List[str]) -> List[str]:
  """
  Divide cada reseña en oraciones con un método robusto para español.

  Args:
      reviews (List[str]): Lista de reseñas.

  Returns:
      List[str]: Lista de todas las oraciones.
  """
  all_sentences = []

  for review in tqdm(reviews, desc="Tokenizando reseñas en oraciones"):
    # Preprocesamiento: asegurar espacios después de puntos, signos de exclamación e interrogación
    # Esto maneja casos como "mala.Manu" -> "mala. Manu"
    review = re.sub(r'([.!?])([A-ZÁÉÍÓÚÑ])', r'\1 \2', review)

    # También manejar casos con comillas y otros signos
    review = re.sub(r'([.!?])\s*"', r'\1 "', review)
    review = re.sub(r'([.!?])\s*\'\'', r'\1 \'\'', review)

    # Usar sent_tokenize con el texto preprocesado
    sentences = sent_tokenize(review, language='spanish')

    # Filtrar oraciones vacías o demasiado cortas
    for sent in sentences:
      sent = sent.strip()
      if len(sent) > 3:  # Ignorar oraciones de menos de 3 caracteres
        all_sentences.append(sent)

  return all_sentences

# Alternativa usando spaCy para segmentación de oraciones (más robusta)
def split_reviews_into_sentences_spacy(reviews: List[str]) -> List[str]:
  """
  Divide cada reseña en oraciones usando spaCy (más robusto).

  Args:
      reviews (List[str]): Lista de reseñas.

  Returns:
      List[str]: Lista de todas las oraciones.
  """
  all_sentences = []

  # Añadir componente de segmentación de oraciones si no existe
  if 'sentencizer' not in nlp.pipe_names:
    nlp.add_pipe('sentencizer')

  for review in tqdm(reviews, desc="Tokenizando reseñas con spaCy"):
    # Procesar el texto con spaCy
    doc = nlp(review)

    # Extraer cada oración
    for sent in doc.sents:
      sent_text = sent.text.strip()
      if len(sent_text) > 3:  # Ignorar oraciones de menos de 3 caracteres
        all_sentences.append(sent_text)

  return all_sentences

# Función híbrida que combina ambos métodos
def split_reviews_into_sentences_hybrid(reviews: List[str]) -> List[str]:
  """
  Divide cada reseña en oraciones usando un método híbrido.

  Args:
      reviews (List[str]): Lista de reseñas.

  Returns:
      List[str]: Lista de todas las oraciones.
  """
  all_sentences = []

  for review in tqdm(reviews, desc="Tokenizando reseñas (método híbrido)"):
    # Preprocesamiento especial para casos problemáticos
    # 1. Separar oraciones pegadas sin espacio después del punto
    review = re.sub(r'([a-záéíóúñ])([.!?])([A-ZÁÉÍÓÚÑ])', r'\1\2 \3', review)

    # 2. Separar oraciones cuando hay un punto y comienza con comillas
    review = re.sub(r'([a-záéíóúñ])([.!?])\s*["\']', r'\1\2 "\'', review)

    # 3. Manejar puntos seguidos de números (no separar en fechas, versiones, etc.)
    # Pero separar cuando es el final de una oración y comienza un número
    review = re.sub(r'([.!?])(\d)', r'\1 \2', review)

    # 4. Separar oraciones que terminan con puntos suspensivos
    review = re.sub(r'([.!?]\.\.)([A-ZÁÉÍÓÚÑ])', r'\1 \2', review)

    # Usar sent_tokenize
    sentences = sent_tokenize(review, language='spanish')

    # Post-procesamiento: dividir oraciones largas que puedan contener múltiples oraciones
    refined_sentences = []
    for sent in sentences:
      # Si la oración es muy larga y tiene puntos internos, intentar dividirla
      if len(sent) > 150 and sent.count('.') > 1:
        # Dividir en puntos que probablemente sean finales de oración
        parts = re.split(r'(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ])', sent)
        for part in parts:
          if part.strip() and len(part.strip()) > 3:
            refined_sentences.append(part.strip())
      else:
        if sent.strip() and len(sent.strip()) > 3:
          refined_sentences.append(sent.strip())

    all_sentences.extend(refined_sentences)

  return all_sentences

# Elegir el método que prefieras:
# Método 1: NLTK con preprocesamiento
# sentences = split_reviews_into_sentences(reviews)

# Método 2: spaCy (más lento pero más preciso)
# sentences = split_reviews_into_sentences_spacy(reviews)

# Método 3: Híbrido (recomendado para balance velocidad/precisión)
sentences = split_reviews_into_sentences_hybrid(reviews)

print(f"Se obtuvieron {len(sentences)} oraciones.")

sentences = [sent.replace('"', '') for sent in sentences]
# Mostrar las primeras 10 oraciones para verificar
print("\nPrimeras 10 oraciones:")
for i, sent in enumerate(sentences[:10]):
  print(f"{i+1}: {sent}")

### Tokenización de Palabras y Procesamiento con SpaCy


In [ ]:
# Tokenización de palabras
tokens_by_sent = {}
for i, sent in enumerate(sentences):
  clean_sent = sent.lower()
  tokens_by_sent[i] = word_tokenize(clean_sent, language='spanish')

# Procesamiento con SpaCy
def process_spacy(texts, batch_size=64):
  return nlp.pipe(texts, batch_size=batch_size)

docs = [
  doc for doc in tqdm(process_spacy(sentences, batch_size=64), total=len(sentences))
]

In [ ]:
idx = 30

for i in range(10):
  print(f"Texto: {docs[idx + i]}")
  print(f"POS: {[token.pos_ for token in docs[idx + i]]}")
  print()

### Extracción de Categorías Gramaticales

In [ ]:
def is_valid_cfg_word(word: str) -> bool:
  """
  Verifica si una palabra es válida para usar en una gramática CFG.
  Las palabras no pueden contener caracteres especiales como *, -, :, ), etc.
  """
  # Patrón de caracteres no permitidos en CFG
  invalid_pattern = r'[*\-:;)({}\[\]<>]'
  return not re.search(invalid_pattern, word)

categories = defaultdict(set)

tags = ["NOUN", "VERB", "AUX", "ADJ", "ADV", "PROPN", "DET", "PRON", "ADP", "CCONJ", "SCONJ", "INTJ", "NUM", "PUNCT"]
# Puede extenderse con: `NOUN__Gender=Masc|Number=Sing`, `NOUN__Gender=Fem|Number=Sing`, `VERB__Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin` 

for doc in docs:
  for token in doc:
    is_found = False
    for tag in tags:
      if token.pos_ == tag:
        word = token.text.lower()
        if is_valid_cfg_word(word):
          categories[tag].add(word)
          is_found = True
    if not is_found:
      print(f"Dont Found: {token.text} | {token.pos_}")
      # categories["OTHER"].add(token.text.lower())

for idx in categories.keys():
  print(f"TAG({idx}) = {categories[idx]}")


In [ ]:
from typing import Hashable

def invert_defaultdict_of_sets(d: DefaultDict[Hashable, Set[Hashable]]) -> DefaultDict[Hashable, Set[Hashable]]:
  inv: DefaultDict[Hashable, Set[Hashable]] = defaultdict(set)
  for k, vs in d.items():
    for v in vs:
      inv[v].add(k)
  return inv

def invert_defaultdict_of_lists(d: DefaultDict[Hashable, List[Hashable]]) -> DefaultDict[Hashable, List[Hashable]]:
  inv: DefaultDict[Hashable, List[Hashable]] = defaultdict(list)
  for k, vs in d.items():
    for v in vs:
      inv[v].append(k)
  return inv

def invert_dict(d: Dict[Hashable, Hashable]) -> Dict[Hashable, Hashable]:
  return {v: k for k, v in d.items()}

def invert_dict_multi(d: Dict[Hashable, Iterable[Hashable]]) -> DefaultDict[Hashable, Set[Hashable]]:
  inv: DefaultDict[Hashable, Set[Hashable]] = defaultdict(set)
  for k, vs in d.items():
    for v in vs:
      inv[v].add(k)
  return inv

inverse_categories = invert_defaultdict_of_sets(categories)
for idx in inverse_categories.keys():
  inverse_categories[idx] = list[Hashable](inverse_categories[idx])[0]
  print(f"WORD({idx}) = TAG({ inverse_categories[idx] })")

In [ ]:
for idx in tokens_by_sent.keys():
  print(f"Doc: {idx+1}: {sentences[idx]}")
  print([ inverse_categories[token] for token in tokens_by_sent[idx] ])

### Definición de la Gramática CFG

In [ ]:
S = """ 
S -> PRELUDE PUNCT CLAUSE PUNCT CLAUSE PUNCT
PRELUDE -> SCONJ NP
CLAUSE -> NP VP
NP -> NP ADP NP CCONJ ADP NP
NP -> DET NOUN | DET NOUN ADJ
NP -> NOUN | PRON | PROPN 
VP -> VERB
VP -> AUX ADV ADJ

S -> CLAUSE PUNCT CLAUSE PUNCT CLAUSE PUNCT CLAUSE PUNCT
CLAUSE -> NP VP | VP NP
CLAUSE -> NP SCONJ VP
NP -> PROPN | PROPN NP
NP -> NOUN | NOUN NP   
NP -> DET NOUN ADJ
VP -> VERB ADP PRON SCONJ VERB
VP -> VERB | VERB ADP ADJ
VP -> ADV VERB NP

S -> NP PUNCT
NP -> PROPN | PROPN PP | PROPN NP PUNCT NP | PROPN NP CCONJ NP
NP -> PP
NP -> DET NP 
NP -> NOUN | NOUN ADJ 
PP -> ADP NP | ADP NP PP

S -> CLAUSE SCONJ NP CLAUSE PUNCT
CLAUSE -> ADP NP | ADV VERB SCONJ VERB
NP -> NOUN ADJ
NP -> NOUN PUNCT | NOUN PUNCT NP | NOUN

S -> NP PP PUNCT PP PUNCT PUNCT
NP -> PRON ADJ
PP -> ADP NPP | ADP NOUN 
NPP -> PROPN PROPN
PUNCT -> COMMA

S -> NP NP VP PUNCT
NP -> NP ADV | NP ADJ
NP -> DET NOUN
VP -> VP CCONJ VP
VP -> PRON VERB PP
PP -> ADP DET NOUN
VP -> PRON VERB PP
PP -> ADP DET NOUN ADJ

S -> NP CLAUSE PUNCT
NP -> ADJ NOUN PP ADJ
PP -> ADP PROPN
CLAUSE -> PRON VP
VP -> VERB VERB NP PP
NP -> DET NOUN
PP -> ADP NP
NP -> DET NOUN ADP NOUN

S -> CLAUSE PUNCT CLAUSE PUNCT
CLAUSE -> NP VP
NP -> DET NOUN
VP -> VERB NOUN
CLAUSE -> NP VP
NP -> DET NOUN
VP -> AUX PP
PP -> ADP NOUN

S -> CLAUSE PUNCT PP CCONJ VERB PUNCT
CLAUSE -> NP VP
NP -> DET PROPN
VP -> VERB NP
NP -> NOUN ADJ ADV NOUN
PP -> ADP NOUN ADJ ADJ

S -> CLAUSE SCONJ CLAUSE PUNCT
CLAUSE -> NP VP PP
NP -> DET NOUN PP
PP -> ADP NOUN
VP -> VERB
PP -> ADP NP
NP -> DET NOUN ADJ ADJ PP
PP -> ADP NOUN PROPN
CLAUSE -> NP ADJ PP ADJ ADV PP PP
NP -> DET NOUN PP
PP -> ADP NOUN
PP -> ADP DET NOUN
PP -> ADP NP
NP -> DET NOUN ADJ
PP -> ADV ADJ SCONJ ADJ

S -> CLAUSE PUNCT
CLAUSE -> NP VP
NP -> DET NOUN
VP -> AUX NP
NP -> DET NOUN ADJ CLAUSE
CLAUSE -> PRON VP
VP -> VERB VERB NP ADJ PP
NP -> DET NOUN ADJ
PP -> ADP NOUN ADJ

S -> NP PUNCT
NP -> ADJ PP
PP -> ADP NOUN

S -> CLAUSE PUNCT
CLAUSE -> PRON VP
VP -> AUX VERB ADJ

S -> ADV CLAUSE PUNCT
CLAUSE -> ADV VP
VP -> AUX PRON ADJ

S -> PP PUNCT
PP -> ADP NP
NP -> DET NOUN

S -> CLAUSE PUNCT CLAUSE
CLAUSE -> NP VP
NP -> PRON
VP -> ADV AUX NP
NP -> DET NOUN ADJ CLAUSE
CLAUSE -> PRON VP
VP -> ADV VERB ADP VERB NP
NP -> DET NOUN
CLAUSE -> CCONJ CLAUSE
CLAUSE -> AUX SCONJ CLAUSE
CLAUSE -> ADV AUX NP
NP -> NOUN ADJ PP
PP -> ADP NOUN

S -> CLAUSE PUNCT CLAUSE PUNCT CLAUSE CCONJ NP PUNCT
CLAUSE -> NP VP
NP -> DET NOUN
NP -> NOUN ADJ
VP -> AUX ADJ
VP -> AUX NP
VP -> PRON VERB VERB PP
PP -> ADP ADV

S -> CLAUSE PUNCT CLAUSE PUNCT CLAUSE PUNCT
CLAUSE -> CCONJ ADV NP | NP VP | SCONJ NP
NP -> PRON ADP NP | DET NOUN PRON VP | DET NUM NOUN ADP ADV | DET PRON
VP -> VERB VERB | VERB CCONJ ADP NOUN

S -> NP PUNCT NP PUNCT Q
Q -> PUNCT NP VP PUNCT
NP -> PROPN | NOUN | DET
VP -> VERB ADJ ADV
"""

In [ ]:
def build_cfg(S, categories):
  lexical_rules = []
  for tag, items in categories.items():
    if items:
      alts = " | ".join(sorted({f"'{w}'" for w in items}))
      lexical_rules.append(f"{tag} -> {alts}")
  
  return "\n".join([S] + lexical_rules)


#head_grammar = ["S -> " + " | ".join([f"S{i+1}" for i in range(len(sentences))])]

grammar_text = build_cfg(S, categories)
grammar = CFG.fromstring(grammar_text)
print(grammar)

### Parsing de Oraciones

In [ ]:
parser = ChartParser(grammar)

parsed_sentences = []
unparsed_sentences = []
parse_trees = []

for i, sent in tqdm(enumerate(sentences), desc="Parseando oraciones", total=len(sentences)):
  toks = tokens_by_sent[i]
  # Filtrar tokens que no sean válidos para el parsing
  filtered_toks = [tok for tok in toks if is_valid_cfg_word(tok)]

  if not filtered_toks:
    unparsed_sentences.append(sent)
    continue

  try:
    trees = list(parser.parse(filtered_toks))
    if trees:
      parsed_sentences.append(sent)
      parse_trees.append(trees)
    else:
      unparsed_sentences.append(sent)
  except Exception as e:
    unparsed_sentences.append(sent)

print(f"Oraciones parseadas: {len(parsed_sentences)}")
print(f"Oraciones no parseadas: {len(unparsed_sentences)}")

# Mostrar ejemplos de oraciones no parseadas
if unparsed_sentences:
  print("\nEjemplos de oraciones no parseadas:")
  for i, sent in enumerate(unparsed_sentences[:5]):
    print(f"{i+1}. {sent}...")

### Extracción de Características

In [ ]:
def extract_tree_features(tree):
  features = {}
  features['tree_depth'] = tree.height()
  features['num_nodes'] = len(list(tree.subtrees()))
  features['num_leaves'] = len(tree.leaves())

  def max_width(t):
    if isinstance(t, str):
      return 1
    widths = [max_width(child) for child in t]
    return max(len(t), max(widths) if widths else 1)

  features['tree_width'] = max_width(tree)

  if features['num_nodes'] > 1:
    features['branching_factor'] = (
        features['num_nodes'] - 1) / (features['num_nodes'] - features['num_leaves'])
  else:
    features['branching_factor'] = 0

  rules_counter = Counter()
  for subtree in tree.subtrees():
    if not isinstance(subtree, str) and len(subtree) > 0:
      rule = f"{subtree.label()} -> {' '.join([child.label() if hasattr(child, 'label') else str(child) for child in subtree])}"
      rules_counter[rule] += 1

  features['unique_rules'] = len(rules_counter)
  features['total_rules'] = sum(rules_counter.values())

  category_counter = Counter()
  for subtree in tree.subtrees():
    if hasattr(subtree, 'label'):
      category_counter[subtree.label()] += 1

  features['categories'] = dict(category_counter)
  return features


def extract_pos_features(tokens, inverse_categories):
  features = {}
  pos_tags = [inverse_categories.get(token, 'UNK') for token in tokens]
  pos_counter = Counter(pos_tags)

  total_tokens = len(tokens)
  features['total_tokens'] = total_tokens

  content_tags = {'NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'}
  function_tags = {'DET', 'ADP', 'CCONJ', 'SCONJ', 'PRON', 'AUX'}

  content_count = sum(pos_counter[tag] for tag in content_tags if tag in pos_counter)
  function_count = sum(pos_counter[tag] for tag in function_tags if tag in pos_counter)

  features['content_words'] = content_count
  features['function_words'] = function_count
  features['lexical_density'] = content_count / total_tokens if total_tokens > 0 else 0

  features['noun_count'] = pos_counter.get('NOUN', 0) + pos_counter.get('PROPN', 0)
  features['verb_count'] = pos_counter.get('VERB', 0) + pos_counter.get('AUX', 0)
  features['adj_count'] = pos_counter.get('ADJ', 0)
  features['adv_count'] = pos_counter.get('ADV', 0)

  features['adj_noun_ratio'] = features['adj_count'] / \
      features['noun_count'] if features['noun_count'] > 0 else 0
  features['adv_verb_ratio'] = features['adv_count'] / \
      features['verb_count'] if features['verb_count'] > 0 else 0
  features['noun_verb_ratio'] = features['noun_count'] / \
      features['verb_count'] if features['verb_count'] > 0 else 0

  unique_tokens = len(set(tokens))
  features['ttr'] = unique_tokens / total_tokens if total_tokens > 0 else 0

  features['subordination_count'] = pos_counter.get('SCONJ', 0)
  features['coordination_count'] = pos_counter.get('CCONJ', 0)

  return features


def calculate_ambiguity_metrics(parse_trees, tokens):
  metrics = {}
  num_parses = len(parse_trees)
  metrics['num_parses'] = num_parses
  metrics['num_tokens'] = len(tokens)
  metrics['ambiguity_ratio'] = num_parses / len(tokens) if len(tokens) > 0 else 0

  if num_parses > 1:
    depths = [tree.height() for tree in parse_trees]
    metrics['parse_depth_variance'] = np.var(depths)
    metrics['parse_depth_mean'] = np.mean(depths)
    metrics['parse_depth_std'] = np.std(depths)
  else:
    metrics['parse_depth_variance'] = 0
    metrics['parse_depth_mean'] = parse_trees[0].height() if parse_trees else 0
    metrics['parse_depth_std'] = 0

  if num_parses > 1:
    metrics['parse_entropy'] = np.log2(num_parses)
  else:
    metrics['parse_entropy'] = 0

  return metrics


# Crear mapeo inverso de categorías
inverse_categories = {}
for tag, words in categories.items():
  for word in words:
    inverse_categories[word] = tag

# Extraer características para todas las oraciones parseadas
all_features = []

for i, sent in enumerate(parsed_sentences):
  if i >= len(parse_trees):
    break

  sent_features = {
      'sentence_id': i,
      'sentence': sent,
      'tokens': tokens_by_sent[i]
  }

  ambiguity_metrics = calculate_ambiguity_metrics(parse_trees[i], tokens_by_sent[i])
  sent_features.update(ambiguity_metrics)

  pos_features = extract_pos_features(tokens_by_sent[i], inverse_categories)
  sent_features.update(pos_features)

  if parse_trees[i]:
    tree_features = extract_tree_features(parse_trees[i][0])
    sent_features.update(tree_features)

    if len(parse_trees[i]) > 1:
      all_tree_features = [extract_tree_features(tree) for tree in parse_trees[i]]
      numeric_keys = ['tree_depth', 'num_nodes', 'num_leaves',
                      'tree_width', 'branching_factor', 'unique_rules', 'total_rules']
      for key in numeric_keys:
        values = [tf[key] for tf in all_tree_features]
        sent_features[f'{key}_mean'] = np.mean(values)
        sent_features[f'{key}_std'] = np.std(values)

  all_features.append(sent_features)

print(f"Características extraídas para {len(all_features)} oraciones parseadas.")

### Análisis Estadístico

In [ ]:
def flatten_features(features_list):
  flattened = []
  for feat in features_list:
    flat_feat = {}
    for key, value in feat.items():
      if key not in ['categories', 'tokens']:
        flat_feat[key] = value
    flattened.append(flat_feat)
  return flattened


if all_features:
  flat_features = flatten_features(all_features)
  df_features = pd.DataFrame(flat_features)

  print("\nEstadísticas descriptivas de las características:")
  numeric_cols = df_features.select_dtypes(include=[np.number]).columns
  print(df_features[numeric_cols].describe())

  # Guardar características en CSV
  df_features.to_csv('features_analysis.csv', index=False, encoding='utf-8')
  print("\nCaracterísticas guardadas en 'features_analysis.csv'")
else:
  print("No hay características para analizar.")

### Almacenar Oraciones No Parseadas

In [ ]:
def save_unparsed_sentences(sentences, file_path):
  with open(file_path, 'w', encoding='utf-8') as f:
    for sent in sentences:
      f.write(sent + '\n')


if unparsed_sentences:
  unparsed_file_path = "unparsed_sentences.txt"
  save_unparsed_sentences(unparsed_sentences, unparsed_file_path)
  print(f"\nOraciones no parseadas guardadas en: {unparsed_file_path}")

  # También guardar información sobre tokens problemáticos
  problem_tokens = set()
  # Analizar solo las primeras 50 para no sobrecargar
  for sent in unparsed_sentences[:50]:
    toks = word_tokenize(sent.lower(), language='spanish')
    for tok in toks:
      if not is_valid_cfg_word(tok):
        problem_tokens.add(tok)

  print(f"\nTokens problemáticos encontrados (primeros 20):")
  for i, tok in enumerate(list(problem_tokens)[:20]):
    print(f"{i+1}. '{tok}'")
else:
  print("\nNo hay oraciones no parseadas.")

### Visualización de Resultados

In [ ]:
if all_features and len(all_features) > 0:
  fig, axes = plt.subplots(2, 2, figsize=(12, 10))

  # Histograma de la profundidad del árbol
  axes[0, 0].hist(df_features['tree_depth'], bins=20, edgecolor='black', alpha=0.7)
  axes[0, 0].set_title('Distribución de la Profundidad del Árbol')
  axes[0, 0].set_xlabel('Profundidad')
  axes[0, 0].set_ylabel('Frecuencia')

  # Scatter plot: Número de tokens vs. Profundidad del árbol
  axes[0, 1].scatter(df_features['total_tokens'], df_features['tree_depth'], alpha=0.5)
  axes[0, 1].set_title('Número de Tokens vs. Profundidad del Árbol')
  axes[0, 1].set_xlabel('Número de Tokens')
  axes[0, 1].set_ylabel('Profundidad del Árbol')

  # Boxplot de la densidad léxica
  if 'lexical_density' in df_features.columns:
    axes[1, 0].boxplot(df_features['lexical_density'].dropna())
    axes[1, 0].set_title('Densidad Léxica')
    axes[1, 0].set_ylabel('Densidad')

  # Histograma del número de parses
  axes[1, 1].hist(df_features['num_parses'], bins=range(
      1, int(df_features['num_parses'].max()) + 2), edgecolor='black', alpha=0.7)
  axes[1, 1].set_title('Distribución del Número de Parses por Oración')
  axes[1, 1].set_xlabel('Número de Parses')
  axes[1, 1].set_ylabel('Frecuencia')

  plt.tight_layout()
  plt.savefig('grammar_analysis_plots.png', dpi=300, bbox_inches='tight')
  plt.show()

  print("Gráficos guardados en 'grammar_analysis_plots.png'")
else:
  print("No hay suficientes datos para visualización.")